<div style="padding:10px ;border:solid 4px #006699; border-radius: 10px; background-color:#CCEEFF;">
    
<table width="100%">
    <tr>
        <td width="10%" style="text-align: left; font-size: large; font-weight: regular; background-color:#CCEEFF;"> 
            Seconde SNT
        </td>
        <td width="80%" style="text-align: center; background-color:#CCEEFF; font-weight: bold;">
            <span style="font-weight: regular; font-size: x-large; ">
                Localisation, cartographie et mobilité
            </span>
            <br/> <br/> 
            <span style="font-weight: bold; font-size: xx-large; ">
                Analyse de données GPS<br/>Décodage de trames NMEA
            </span>
            <br/>
        </td>
        <td style="background-color:#CCEEFF;" > 
            <img src ="https://capytale2.ac-paris.fr/web/sites/default/files/2024/09-24/21-49-57/icone-gps.png" width="100%" height="100%" > 
        </td>
    </tr>
</table>
</div>

<style>
    html {
        font-family: Serif;
    }
    h1, h2 {
        color: #006699;
        margin-bottom: 20px;
    }
    .title-number { 
    }
    .title-number:before {
        content: "███";
        margin-right: 20px;
    }
    .title-number:after {
        content: "▏";
        margin-left: 20px;
        margin-right: -15px;
    }
    .question {
        font-weight: bold;
    }
</style>

<h1><span class="title-number">I</span>Contexte</h1>

_Dans ce TP, nous allons voir concrètement comment les informations sont envoyées par les satellites jusqu'à notre téléphone._

Le système de géolocalisation permet de se positionner sur Terre. Une fois la position calculée grâce à plusieurs satellites (par le biais de la trilatération), la puce GPS de notre récepteur (le smartphone, par exemple) transmet ces coordonnées à l'application de cartographie.

La norme la plus couramment utilisée pour transférer ces données s'appelle **NMEA 0183** (*National Marine Electronics Association*).

Pour le processeur et la puce GPS, voici à quoi ressemble « une phrase » (que l'on nomme **une trame NMEA**) : 

`$GPGGA,064036.000,4836.5375,N,00740.9373,E,1,04,3.2,200.2,M,,,,0000*0E`

À première vue, cela ressemble à une suite de caractères incompréhensibles... Et pourtant, tout suit un fonctionnement très précis ! Remarquez la présence des **virgules** : chaque information (ou **champ**) est bien séparée de la précédente.

<h1><span class="title-number">II</span>Structure d'une trame GPGGA</h1>

Il existe plusieurs types de trames NMEA. Celles commençant par `$GPGGA` sont les plus fréquentes car elles contiennent l'essentiel : **l'heure de la mesure, la position (latitude, longitude) et l'altitude**.

Étudions de plus près notre exemple : 

| Position/Indice | Donnée | Valeur ici | Explications |
| :--- | :--- | :--- | :--- |
| **0** | En-tête | `$GPGGA` | Précise le type de la trame de données |
| **1** | Heure UTC| `064036.000` | L'heure à Greenwich (06h 40min 36,000s) |
| **2** | Latitude | `4836.5375` | 48° et 36.5375' (Degrés et Minutes d'arc) |
| **3** | Hémisphère| `N` | Nord (N) ou Sud (S) |
| **4** | Longitude| `00740.9373` | 007° et 40.9373' (Degrés et Minutes d'arc) |
| **5** | Hémisphère| `E` | Est (E) ou Ouest (W) |
| **6** | Qualité | `1` | Mode (1 = GPS fixe) |
| **7** | Satellites| `04` | Nombre de satellites utilisés (ici 4) |
| **8** | Précision| `3.2` | Précision horizontale |
| **9** | Altitude | `200.2` | Altitude de l'antenne |
| **10** | Unité | `M` | M (pour Mètres) |

_Note : Les champs restants à la fin de la trame (notamment le `*0E`) servent de sécurité (checksum) pour vérifier qu'il n'y a pas eu d'erreur de transmission._

!!! question Question 1
Voici une nouvelle trame NMEA interceptée par un autre smartphone : 

`$GPGGA,123519.000,4851.5040,N,00217.6700,E,1,08,1.03,61.7,M,47.3,M,,*6C`

En lisant bien les champs, entre les virgules de cette nouvelle trame, indiquer : 
1. L'heure de la mesure (Heure UTC) 
2. Le nombre de satellites qui ont été utilisés 
3. L'altitude en mètres
!!!

<h1><span class="title-number">III</span>Traitement par programme (Python)</h1>

Plutôt que de devoir lire manuellement chaque ligne et compter les virgules avec le doigt, nous allons utiliser le langage **Python** car l'ordinateur peut récupérer ces données en un clin d'œil.

En Python, une simple fonction nommée `split()` (« diviser » en anglais) permet de séparer notre trame de texte en une **liste de mots**, en coupant à chaque fois qu'il trouve une `","` !

!!! question Question 2
Exécuter la cellule contenant le code ci-dessous.
Observez en particulier de quelle manière nous pouvons extraire l'heure (champ situé à l'indice **1** de notre liste de mots).
!!!

In [ ]:
# ============================================================
# ETAPE 1 : Decoupage de la trame NMEA en Python
# ============================================================

# On stocke notre longue trame dans une variable nommée 'trame'
trame = "$GPGGA,123519.000,4851.5040,N,00217.6700,E,1,08,1.03,61.7,M,47.3,M,,*6C"

# On decoupe la chaine à l'aide des virgules. 
# Le resultat sera une donnee de type "liste" (tableau)
liste_des_champs = trame.split(",")

print("La liste complete :", liste_des_champs)
print("--------------------------------------------------")

# Attention, on rappelle qu'en Python, on commence a compter a ZERO !
print("Le champ en indice 0 est l'en-tete : ", liste_des_champs[0])

# Pour acceder a l'heure, qui est le 2e champ, on donne donc l'indice 1 !
heure_utc = liste_des_champs[1]
print("Le champ en indice 1 correspond a l'heure : ", heure_utc)

!!! question Question 3
Dans la cellule ci-dessous, en te basant sur le tableau de la partie **II**, complète le code (les `...`) pour afficher la *latitude*, la *longitude*, l'*altitude* et le *nombre de satellites* interceptés.
!!!

In [ ]:
# ============================================================
# QUESTION 3 : Extraire la latitude, la longitude, etc.
# ------------------------------------------------------------
# Utiliser la bonne position (indice numerique) entre les 
# crochets pour cibler l'information souhaitee.
# ============================================================

# La latitude
latitude = liste_des_champs[...]
print("Latitude  : ", latitude)

# La longitude
longitude = liste_des_champs[...]
print("Longitude : ", longitude)

# L'altitude 
altitude = liste_des_champs[...]
print("Altitude  : ", altitude, " m")

# Le nombre de satellites
satellites = liste_des_champs[...]
print("Satellites: ", satellites)

<h1><span class="title-number">IV</span>Conversion des coordonnées</h1>

Les coordonnées issues du NMEA (la latitude `4851.5040` par exemple) sont difficiles à transmettre à un module de cartographie car elles sont écrites dans le système sexagésimal (Degrés, Minutes d'arc).

Dans le format NMEA pour la latitude **`DDMM.MMMM`** :
- **48** représente les degrés (`DD`)
- **51.5040** représente les minutes d'arc (`MM.MMMM`)

Sachant que $1 \text{ degré} = 60 \text{ minutes}$, la conversion mathématique pour obtenir une **valeur complètement décimale** (celle qu'on retrouve sur Google Maps par exemple) est :

$$ \text{Valeur decimale} = \text{Degrés} + \frac{\text{Minutes}}{60} $$
 
Pour la valeur `4851.5040`, en faisant le calcul complet cela donne : 

$48 + \frac{51.5040}{60} = 48 + 0.8584 = 48.8584$ (en degrés continus).

!!! question Question 4
Nous avons fabriqué ci-dessous une fonction complète en Python nommée `conversion_vers_decimal(...)` qui permet d'automatiser tout ce calcul pour n'importe quelle coordonnée NMEA.
Lire attentivement les commentaires et exécuter la cellule.
!!!

In [ ]:
def conversion_vers_decimal(valeur_nmea):
    """
    Cette fonction convertit une donnee NMEA 'GGMM.MMMM' ou 'GGGMM.MMMM'
    en un nombre decimal complet (degres classiques).
    """
    # 1. On trouve la position du "." dans la chaine de texte
    index_point = valeur_nmea.find(".")
    
    # 2. Les degres se trouvent AVANT "MM.MMMM", soit exactement 2 positions avant le point.
    # Cela permet de separer (par expl dans '4851.5040' : degres_txt deviendra '48') 
    degres_txt = valeur_nmea[:index_point - 2]
    
    # 3. Les minutes sont situees apres (c'est a dire '51.5040')
    minutes_txt = valeur_nmea[index_point - 2:]
    
    # 4. On convertit ensuite nos bouts de phrases en valeurs numeriques à virgule flottante (float)
    degres = float(degres_txt)
    minutes = float(minutes_txt)
    
    # 5. On applique la fameuse formule de conversion !
    valeur_decimale = degres + (minutes / 60)
    
    # On s'assure d'arrondir à 4 chiffres apres la virgule pour la lisibilite
    return round(valeur_decimale, 4)


# --- TEST SUR LA LATITUDE --- 
lat_decimale = conversion_vers_decimal("4851.5040")
print("Apres conversion, la latitude decimale vaut :", lat_decimale)


!!! question Question 5
En t'inspirant de la dernière ligne du programme précédent, complète le code ci-dessous pour lancer la conversion de la **longitude** (à savoir : la chaîne `"00217.6700"` extraite à la question 3).
!!!

In [ ]:
# ============================================================
# QUESTION 5 : Convertir la longitude extraite par script
# ============================================================

# A COMPLETER : Utilise la fonction avec la valeur qu'il faut.
lon_decimale = ...( ... )

print("Apres conversion, la longitude decimale vaut :", lon_decimale)


<h1><span class="title-number">V</span>Sur la carte !</h1>

Grâce aux questions précédentes, nous avons pu automatiser intelligemment le décryptage d'une trame et la création de coordonnées décimales valides, propres au format de cartographie informatique standard.

Nous avons :
- **Latitude** Décimale (Nord, donc valeur positive) : `+ 48.8584` 
- **Longitude** Décimale (Est, donc valeur positive): `+ 2.2945`

!!! question Question 6
Dans la cellule ci-dessous, à l'aide de la bibliothèque `folium`, exécuter le code pour centrer la carte mondiale directement sur les coordonnées (GPS) de notre mesure issue de la Trame NMEA et l'identifier.
!!!

In [ ]:
# ============================================================
# QUESTION 6 : Affichage sur carte de notre acquisition
# ============================================================
import folium

# On definit la position avec nos latitudes et longitudes converties pretes a l'emploi
position_gps = [48.8584, 2.2945]

# On cree la carte centree sur cette position avec un fort zoom
ma_carte = folium.Map(location=position_gps, zoom_start=17)

# On ajoute le marqueur a notre carte pour bien visualiser
folium.Marker(
    location=position_gps,
    tooltip="Position du recepteur de la Trame NMEA",
    icon=folium.Icon(color='purple', icon='flag')
).add_to(ma_carte)

# Affichage effectif de la carte (patienter un petit peu!)
ma_carte


!!! question Question 7
Dézoomer légèrement la carte avec la souris (molette) ou le bouton `-`... Qui a émis d'un seul coup cette Trame NMEA ? Quel est le monument mondialement connu de cet endroit ?
!!!